# ETAPA 3b — Análise Triple-DiD: IA Generativa, Conectividade e Emprego Formal

**Dissertação:** Inteligência Artificial Generativa e o Mercado de Trabalho Brasileiro.

**Objetivo:** Estimar o efeito diferencial da IA generativa entre municípios de alta e baixa conectividade (Triple-DiD). Unidade: ocupação × município × mês. Coeficiente de interesse: **β₇** (triple_did = post × alta_exp × alta_conectividade).

**Input:** `data/output/painel_caged_municipio_anatel.parquet` (Notebook 3a).

### Estratégia de identificação

| Elemento | Especificação |
|----------|---------------|
| Unidade | Ocupação (CBO 4d) × Município × Mês |
| Tratamento (1) | Alta exposição à IA (ILO score > mediana) |
| Tratamento (2) | Alta conectividade (penetração BL > mediana) |
| Evento | Lançamento ChatGPT (Nov/2022) |
| FE | cbo_4d + uf_periodo |
| Clustering | id_municipio |
| Coef. interesse | β₇ (triple_did) |

### 1. Configuração e carga de dados

Carregar painel 3a, winsorizar salários (P1/P99).

In [1]:
# Etapa 3b.1 — Configuração e carga
import warnings
import pandas as pd
import numpy as np
import pyfixest as pf
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*multicollinearity.*", category=UserWarning)

DATA_OUTPUT = Path("data/output")
OUTPUTS_TABLES = Path("outputs/tables")
OUTPUTS_FIGURES = Path("outputs/figures")
for d in [OUTPUTS_TABLES, OUTPUTS_FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

PAINEL_FILE = DATA_OUTPUT / "painel_caged_municipio_anatel.parquet"
PAINEL_FILE_V2 = DATA_OUTPUT / "painel_caged_municipio_anatel_v2.parquet"
VCOV_SPEC = {"CRV1": "id_municipio"}
OUTCOMES = {"ln_salario_real_adm": "Log(Sal. Real Adm.)", "ln_admissoes": "Log(Admissões)", "pct_superior_adm": "% Superior", "idade_media_adm": "Idade Média"}

if PAINEL_FILE_V2.exists():
    df = pd.read_parquet(PAINEL_FILE_V2)
    print("Painel v2 (faixa etária, fibra, capital) carregado.")
else:
    df = pd.read_parquet(PAINEL_FILE)
for c in ["salario_medio_adm", "salario_real_adm"]:
    if c in df.columns:
        lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
        df[c] = df[c].clip(lo, hi)
if "ln_salario_real_adm" in df.columns:
    df["ln_salario_real_adm"] = np.log(df["salario_real_adm"].clip(lower=1))
print(f"Painel: {len(df):,} obs | {df['id_municipio'].nunique():,} municípios | {df['cbo_4d'].nunique()} ocupações")

Painel: 5,534,808 obs | 657 municípios | 614 ocupações


### 2. Tabela de balanço por conectividade (pré-tratamento)

In [2]:
# Etapa 3b.2 — Balanço
pre = df[df["post"] == 0]
print(pre.groupby("alta_conectividade").agg(
    n_obs=("cbo_4d", "count"),
    admissoes_media=("admissoes", "mean"),
    salario_medio=("salario_medio_adm", "mean"),
    penetracao_media=("penetracao_bl", "mean"),
).round(2))

                      n_obs  admissoes_media  salario_medio  penetracao_media
alta_conectividade                                                           
0                    205077             5.02        1330.58              0.18
1                   2126257            15.02        1742.66              0.69


### 3. DiD por subgrupo de conectividade (motivação)

Estimar DiD (post × alta_exp) separadamente para municípios de alta e baixa conectividade.

In [3]:
# Etapa 3b.3 — DiD por subgrupo
df["post_alta_exp"] = df["post"] * df["alta_exp"]
for conn_label, mask in [("Alta conect.", df["alta_conectividade"] == 1), ("Baixa conect.", df["alta_conectividade"] == 0)]:
    d = df.loc[mask].dropna(subset=["ln_salario_real_adm"])
    m = pf.feols("ln_salario_real_adm ~ post_alta_exp | cbo_4d + uf_periodo", data=d, vcov=VCOV_SPEC)
    c = m.coef().get("post_alta_exp", m.coef().iloc[0])
    print(f"{conn_label}: coef = {float(c):.4f}, se = {float(m.se().loc['post_alta_exp']):.4f}")

Alta conect.: coef = -0.0058, se = 0.0018
Baixa conect.: coef = -0.0089, se = 0.0046


### 4. Triple-DiD — Modelo principal

outcome ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo

In [4]:
# Etapa 3b.4 — Triple-DiD principal
formula = "ln_salario_real_adm ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo"
m = pf.feols(formula, data=df.dropna(subset=["ln_salario_real_adm"]), vcov=VCOV_SPEC)
print(m.summary())
results_triple = pd.DataFrame({"coef": m.coef(), "se": m.se(), "p_value": m.pvalue()})
results_triple.to_csv(OUTPUTS_TABLES / "triple_did_main_etapa3b.csv")

###

Estimation:  OLS
Dep. var.: ln_salario_real_adm, Fixed effects: cbo_4d+uf_periodo
Inference:  CRV1
Observations:  5534808

| Coefficient          |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| triple_did           |     -0.016 |        0.008 |    -1.888 |      0.059 | -0.032 |   0.001 |
| post_alta_exp        |      0.008 |        0.008 |     1.069 |      0.285 | -0.007 |   0.024 |
| post_alta_conect     |      0.017 |        0.008 |     2.239 |      0.025 |  0.002 |   0.032 |
| alta_exp_alta_conect |      0.037 |        0.010 |     3.743 |      0.000 |  0.017 |   0.056 |
---
RMSE: 0.555 R2: 0.969 R2 Within: 0.0 
None


### 5. Event study por grupo de conectividade

Dummies (tempo_relativo × alta_exp) para alta e baixa conectividade; referência t = -1.

In [8]:
# Etapa 3b.5 — Event study por conectividade
# Dummies (tempo_relativo × alta_exp) para alta e baixa conectividade; referência t = -1.
# Nomes das variáveis: sem "-" na fórmula (formulaic interpreta "did_t-12" como did_t menos 12).
def _nome_did(t):
    return f"did_m{-t}" if t < 0 else f"did_{t}"

BIN_MIN, BIN_MAX, REF = -12, 24, -1
df["t_bin"] = df["tempo_relativo_meses"].clip(lower=BIN_MIN, upper=BIN_MAX)
for conn_val, label in [(1, "Alta"), (0, "Baixa")]:
    d = df[(df["alta_conectividade"] == conn_val) & df["ln_salario_real_adm"].notna()].copy()
    ts = [t for t in sorted(d["t_bin"].unique()) if t != REF]
    did_vars = [_nome_did(t) for t in ts]
    for t, v in zip(ts, did_vars):
        d[v] = ((d["t_bin"] == t) & (d["alta_exp"] == 1)).astype(int)
    if did_vars:
        m = pf.feols(f"ln_salario_real_adm ~ {' + '.join(did_vars)} | cbo_4d + uf_periodo", data=d, vcov=VCOV_SPEC)
        print(f"{label} conect.: {len(ts)} coeficientes estimados")

Alta conect.: 36 coeficientes estimados
Baixa conect.: 36 coeficientes estimados


### 6. Testes de robustez e síntese

Placebo temporal (Dez/2021); cutoff Q75. Correção FDR para múltiplos outcomes (opcional). Síntese: comparar com Etapa 2 — efeito concentrado onde a adoção de IA é viável.

In [9]:
# Etapa 3b.6 — Robustez (placebo) e múltiplos outcomes
# Placebo: post = 1 a partir de Dez/2021
df["post_placebo"] = ((df["ano"] == 2021) & (df["mes"] >= 12)) | (df["ano"] > 2021)
df["triple_did_placebo"] = df["post_placebo"].astype(int) * df["alta_exp"] * df["alta_conectividade"]
df["post_alta_exp_placebo"] = df["post_placebo"].astype(int) * df["alta_exp"]
m_placebo = pf.feols("ln_salario_real_adm ~ triple_did_placebo + post_alta_exp_placebo | cbo_4d + uf_periodo",
                     data=df.dropna(subset=["ln_salario_real_adm"]), vcov=VCOV_SPEC)
print("Placebo Dez/2021 (triple_did_placebo):", m_placebo.coef().get("triple_did_placebo"), "p =", m_placebo.pvalue().get("triple_did_placebo"))
print("Esperado: não significativo.")

Placebo Dez/2021 (triple_did_placebo): 0.03653039627709792 p = 5.672064404560473e-05
Esperado: não significativo.


### 7. Correção de tendência diferencial pré (Caminho 1)

Se o placebo falha por tendência pré-existente no grupo alta_exp × alta_conectividade, controlar por **alta_exp × alta_conectividade × trend** (trend = 1, 2, …, T). Se β₇ (triple_did) permanece significativo após incluir a tendência, o efeito é robusto; se não, o efeito era explicado pela tendência pré.

In [ ]:
# Etapa 3b.7 — Tendência diferencial pré (Caminho 1)
periodos_ordenados = sorted(df["periodo"].unique())
trend_map = {p: i + 1 for i, p in enumerate(periodos_ordenados)}
df["trend"] = df["periodo"].map(trend_map)
df["trend_exp_conect"] = df["alta_exp"] * df["alta_conectividade"] * df["trend"]
df["trend_exp"] = df["alta_exp"] * df["trend"]
df["trend_conect"] = df["alta_conectividade"] * df["trend"]

outcome = "ln_salario_real_adm"
formula_trend = (
    f"{outcome} ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect"
    " + trend_exp_conect + trend_exp + trend_conect | cbo_4d + uf_periodo"
)
model_trend = pf.feols(formula_trend, data=df.dropna(subset=[outcome]), vcov=VCOV_SPEC)
print("Modelo com tendência diferencial pré:")
print(model_trend.summary())

# Comparar com modelo original (seção 4)
formula_orig = "ln_salario_real_adm ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo"
model_orig = pf.feols(formula_orig, data=df.dropna(subset=["ln_salario_real_adm"]), vcov=VCOV_SPEC)
print("\nComparação: Original vs. com tendência")
pf.etable([model_orig, model_trend])

### 8. Proxies alternativos de conectividade (Caminho 2)

Testar quatro proxies além da mediana de penetração: (1) extremos Q75 vs Q25, (2) % fibra óptica, (3) tratamento contínuo (dose-resposta), (4) apenas capitais. Para cada um, rodar Triple-DiD e placebo temporal (Dez/2021). Se algum proxy passar no placebo, sugere que o problema está na definição de conectividade.

In [ ]:
# Etapa 3b.8 — Proxies alternativos de conectividade
outcome = "ln_salario_real_adm"
resultados_proxy = []

# (1) Original (mediana penetração) — já estimado na seção 4
formula_orig = "ln_salario_real_adm ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo"
m_orig = pf.feols(formula_orig, data=df.dropna(subset=[outcome]), vcov=VCOV_SPEC)
resultados_proxy.append({"proxy": "Original (mediana)", "coef": m_orig.coef().get("triple_did"), "se": m_orig.se().get("triple_did"), "pval": m_orig.pvalue().get("triple_did"), "placebo_pval": m_placebo.pvalue().get("triple_did_placebo")})

# (2) Extremos Q75 vs Q25
q25, q75 = df["penetracao_bl"].quantile(0.25), df["penetracao_bl"].quantile(0.75)
df_extremos = df[(df["penetracao_bl"] <= q25) | (df["penetracao_bl"] >= q75)].copy()
df_extremos["alta_conect_extremo"] = (df_extremos["penetracao_bl"] >= q75).astype(int)
df_extremos["triple_did_extremo"] = df_extremos["post"] * df_extremos["alta_exp"] * df_extremos["alta_conect_extremo"]
df_extremos["post_alta_conect_ext"] = df_extremos["post"] * df_extremos["alta_conect_extremo"]
df_extremos["alta_exp_alta_conect_ext"] = df_extremos["alta_exp"] * df_extremos["alta_conect_extremo"]
formula_ext = f"{outcome} ~ triple_did_extremo + post_alta_exp + post_alta_conect_ext + alta_exp_alta_conect_ext | cbo_4d + uf_periodo"
m_ext = pf.feols(formula_ext, data=df_extremos.dropna(subset=[outcome]), vcov=VCOV_SPEC)
df_extremos["post_placebo"] = ((df_extremos["ano"] == 2021) & (df_extremos["mes"] >= 12)) | (df_extremos["ano"] > 2021)
df_extremos["triple_placebo_ext"] = df_extremos["post_placebo"].astype(int) * df_extremos["alta_exp"] * df_extremos["alta_conect_extremo"]
df_extremos["post_placebo_alta_exp"] = df_extremos["post_placebo"].astype(int) * df_extremos["alta_exp"]
df_extremos["post_placebo_alta_conect_ext"] = df_extremos["post_placebo"].astype(int) * df_extremos["alta_conect_extremo"]
m_placebo_ext = pf.feols(f"{outcome} ~ triple_placebo_ext + post_placebo_alta_exp + post_placebo_alta_conect_ext + alta_exp_alta_conect_ext | cbo_4d + uf_periodo", data=df_extremos.dropna(subset=[outcome]), vcov=VCOV_SPEC)
try:
    pval_placebo_ext = m_placebo_ext.pvalue().get("triple_placebo_ext", np.nan)
except Exception:
    pval_placebo_ext = np.nan
resultados_proxy.append({"proxy": "Extremos Q75/Q25", "coef": m_ext.coef().get("triple_did_extremo"), "se": m_ext.se().get("triple_did_extremo"), "pval": m_ext.pvalue().get("triple_did_extremo"), "placebo_pval": pval_placebo_ext})

# (3) % Fibra (requer painel v2)
if "triple_did_fibra" in df.columns:
    formula_fibra = f"{outcome} ~ triple_did_fibra + post_alta_exp + post_alta_fibra + alta_exp_alta_fibra | cbo_4d + uf_periodo"
    m_fibra = pf.feols(formula_fibra, data=df.dropna(subset=[outcome]), vcov=VCOV_SPEC)
    df["triple_placebo_fibra"] = df["post_placebo"].astype(int) * df["alta_exp"] * df["alta_fibra"]
    df["post_placebo_alta_fibra"] = df["post_placebo"].astype(int) * df["alta_fibra"]
    m_pl_fibra = pf.feols(f"{outcome} ~ triple_placebo_fibra + post_alta_exp_placebo + post_placebo_alta_fibra + alta_exp_alta_fibra | cbo_4d + uf_periodo", data=df.dropna(subset=[outcome]), vcov=VCOV_SPEC)
    resultados_proxy.append({"proxy": "% Fibra", "coef": m_fibra.coef().get("triple_did_fibra"), "se": m_fibra.se().get("triple_did_fibra"), "pval": m_fibra.pvalue().get("triple_did_fibra"), "placebo_pval": m_pl_fibra.pvalue().get("triple_placebo_fibra")})
else:
    resultados_proxy.append({"proxy": "% Fibra", "coef": np.nan, "se": np.nan, "pval": np.nan, "placebo_pval": np.nan})

# (4) Contínuo (dose-resposta)
df["dose_triple"] = df["penetracao_bl"] * df["ilo_exposure_score"] * df["post"] if "ilo_exposure_score" in df.columns else df["penetracao_bl"] * df["alta_exp"].astype(float) * df["post"]
exp_col = "ilo_exposure_score" if "ilo_exposure_score" in df.columns else "alta_exp"
df["dose_exp_post"] = (df[exp_col].astype(float) if exp_col == "alta_exp" else df[exp_col]) * df["post"]
df["dose_conect_post"] = df["penetracao_bl"] * df["post"]
df["dose_exp_conect"] = (df[exp_col].astype(float) if exp_col == "alta_exp" else df[exp_col]) * df["penetracao_bl"]
formula_cont = f"{outcome} ~ dose_triple + dose_exp_post + dose_conect_post + dose_exp_conect | cbo_4d + uf_periodo"
m_cont = pf.feols(formula_cont, data=df.dropna(subset=[outcome]), vcov=VCOV_SPEC)
resultados_proxy.append({"proxy": "Contínuo", "coef": m_cont.coef().get("dose_triple"), "se": m_cont.se().get("dose_triple"), "pval": m_cont.pvalue().get("dose_triple"), "placebo_pval": np.nan})

# (5) Capitais (requer painel v2)
if "capital" in df.columns:
    df["triple_did_capital"] = df["post"] * df["alta_exp"] * df["capital"]
    df["post_capital"] = df["post"] * df["capital"]
    df["alta_exp_capital"] = df["alta_exp"] * df["capital"]
    formula_cap = f"{outcome} ~ triple_did_capital + post_alta_exp + post_capital + alta_exp_capital | cbo_4d + uf_periodo"
    try:
        m_cap = pf.feols(formula_cap, data=df.dropna(subset=[outcome]), vcov=VCOV_SPEC)
        resultados_proxy.append({"proxy": "Capitais", "coef": m_cap.coef().get("triple_did_capital"), "se": m_cap.se().get("triple_did_capital"), "pval": m_cap.pvalue().get("triple_did_capital"), "placebo_pval": np.nan})
    except Exception:
        resultados_proxy.append({"proxy": "Capitais", "coef": np.nan, "se": np.nan, "pval": np.nan, "placebo_pval": np.nan})
else:
    resultados_proxy.append({"proxy": "Capitais", "coef": np.nan, "se": np.nan, "pval": np.nan, "placebo_pval": np.nan})

tab_proxy = pd.DataFrame(resultados_proxy)
tab_proxy["sig"] = tab_proxy["pval"].apply(lambda p: "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else "")))
print("Tabela comparativa — Triple-DiD por proxy de conectividade")
print(tab_proxy.to_string())

### 9. Decomposição etária do Triple-DiD

Rodar Triple-DiD para cada outcome por faixa etária (admissões e salário por jovem/intermediário/senior, share de jovens/seniores, razão salarial jovem/senior). Se a hipótese de compensação etária estiver correta: share_jovem ↓, share_senior ↑, ln_sal_real_jovem ↓, razao_sal_jovem_senior ↓.

In [ ]:
# Etapa 3b.9 — Decomposição etária do Triple-DiD
outcomes_idade = {
    "Admissões jovens (log)": "ln_adm_jovem",
    "Admissões intermediários (log)": "ln_adm_intermediario",
    "Admissões seniores (log)": "ln_adm_senior",
    "Share jovens nas admissões": "share_jovem",
    "Share seniores nas admissões": "share_senior",
    "Salário real jovens (log)": "ln_sal_real_jovem",
    "Salário real intermediários (log)": "ln_sal_real_intermediario",
    "Salário real seniores (log)": "ln_sal_real_senior",
    "Razão salarial jovem/senior": "razao_sal_jovem_senior",
}

resultados_idade = {}
formula_base = "{outcome} ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo"
for nome, outcome_var in outcomes_idade.items():
    if outcome_var not in df.columns:
        print(f"SKIP {nome}: coluna {outcome_var} não encontrada (rode 3a com painel v2).")
        continue
    df_valid = df.dropna(subset=[outcome_var])
    if len(df_valid) < 1000:
        print(f"SKIP {nome}: apenas {len(df_valid)} obs válidas")
        continue
    formula = formula_base.format(outcome=outcome_var)
    try:
        model = pf.feols(formula, data=df_valid, vcov=VCOV_SPEC)
        coef = model.coef().get("triple_did")
        se = model.se().get("triple_did")
        pval = model.pvalue().get("triple_did")
        resultados_idade[nome] = {"coef": coef, "se": se, "pval": pval, "n": len(df_valid)}
    except Exception as e:
        print(f"ERRO {nome}: {e}")

if resultados_idade:
    df_resultados_idade = pd.DataFrame(resultados_idade).T
    df_resultados_idade["sig"] = df_resultados_idade["pval"].apply(
        lambda p: "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else ""))
    )
    print("Triple-DiD por outcome etário (coef = triple_did):")
    print(df_resultados_idade.to_string())
else:
    print("Nenhum outcome etário disponível. Execute o Notebook 3a e exporte o painel v2.")

### 10. Modelo reformulado: impacto sobre jovens

Hipótese: após o ChatGPT, a **participação de jovens nas contratações** (share_jovem) caiu mais em ocupações expostas à IA e em municípios conectados. Outcome principal: **share_jovem** = admissões de jovens (&lt;30) / total de admissões. Três especificações (pura, com tendência, com controles demográficos), placebo temporal e event study por conectividade.

In [ ]:
# Etapa 3b.10 — Modelo reformulado: share_jovem
if "share_jovem" not in df.columns:
    print("share_jovem não disponível. Execute o Notebook 3a e exporte o painel v2.")
else:
    df_j = df.dropna(subset=["share_jovem"]).copy()
    # Garantir variáveis de tendência (seção 7)
    if "trend_exp_conect" not in df_j.columns and "periodo" in df_j.columns:
        periodos_ordenados = sorted(df_j["periodo"].unique())
        trend_map = {p: i + 1 for i, p in enumerate(periodos_ordenados)}
        df_j["trend"] = df_j["periodo"].map(trend_map)
        df_j["trend_exp_conect"] = df_j["alta_exp"] * df_j["alta_conectividade"] * df_j["trend"]
        df_j["trend_exp"] = df_j["alta_exp"] * df_j["trend"]
        df_j["trend_conect"] = df_j["alta_conectividade"] * df_j["trend"]

    formula_j1 = "share_jovem ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo"
    m_jovem_1 = pf.feols(formula_j1, data=df_j, vcov=VCOV_SPEC)

    formula_j2 = "share_jovem ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect + trend_exp_conect + trend_exp + trend_conect | cbo_4d + uf_periodo"
    m_jovem_2 = pf.feols(formula_j2, data=df_j, vcov=VCOV_SPEC)

    controles = []
    if "pct_superior_adm" in df_j.columns:
        controles.append("pct_superior_adm")
    if "pct_mulher_adm" in df_j.columns:
        controles.append("pct_mulher_adm")
    if "ln_pib_pc" in df_j.columns:
        controles.append("ln_pib_pc")
    formula_j3 = "share_jovem ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect + trend_exp_conect + trend_exp + trend_conect + " + " + ".join(controles) + " | cbo_4d + uf_periodo" if controles else formula_j2
    m_jovem_3 = pf.feols(formula_j3, data=df_j.dropna(subset=controles) if controles else df_j, vcov=VCOV_SPEC)

    print("Três especificações — outcome: share_jovem")
    pf.etable([m_jovem_1, m_jovem_2, m_jovem_3])

    # Placebo temporal (Dez/2021) para share_jovem
    df_pre = df[df["periodo_dt"] < "2022-11-01"].copy() if "periodo_dt" in df.columns else df[(df["ano"] < 2022) | ((df["ano"] == 2022) & (df["mes"] < 11))].copy()
    if "periodo_dt" not in df_pre.columns and "ano" in df_pre.columns:
        df_pre["periodo_dt"] = pd.to_datetime(df_pre["ano"].astype(str) + "-" + df_pre["mes"].astype(str).str.zfill(2) + "-01", errors="coerce")
    df_pre["post_placebo"] = (df_pre["periodo_dt"] >= "2021-12-01").astype(int) if "periodo_dt" in df_pre.columns else ((df_pre["ano"] == 2021) & (df_pre["mes"] >= 12)) | (df_pre["ano"] > 2021)
    df_pre["triple_placebo"] = df_pre["post_placebo"].astype(int) * df_pre["alta_exp"] * df_pre["alta_conectividade"]
    df_pre["post_placebo_alta_exp"] = df_pre["post_placebo"].astype(int) * df_pre["alta_exp"]
    df_pre["post_placebo_alta_conect"] = df_pre["post_placebo"].astype(int) * df_pre["alta_conectividade"]
    df_pre = df_pre.dropna(subset=["share_jovem"])
    m_placebo_j = pf.feols("share_jovem ~ triple_placebo + post_placebo_alta_exp + post_placebo_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo", data=df_pre, vcov=VCOV_SPEC)
    print(f"Placebo share_jovem (Dez/2021): coef = {m_placebo_j.coef().get('triple_placebo', np.nan):.4f}, p = {m_placebo_j.pvalue().get('triple_placebo', np.nan):.4f}")

    # Event study para share_jovem por grupo de conectividade (mesma lógica da seção 5)
    if "tempo_relativo_meses" in df.columns:
        BIN_MIN, BIN_MAX, REF = -12, 24, -1
        df_es = df.dropna(subset=["share_jovem"]).copy()
        df_es["t_bin"] = df_es["tempo_relativo_meses"].clip(lower=BIN_MIN, upper=BIN_MAX)
        for conn_val, label in [(1, "Alta"), (0, "Baixa")]:
            d = df_es[(df_es["alta_conectividade"] == conn_val)].copy()
            ts = [t for t in sorted(d["t_bin"].unique()) if t != REF]
            did_vars = [f"did_m{-t}" if t < 0 else f"did_{t}" for t in ts]
            for t, v in zip(ts, did_vars):
                d[v] = ((d["t_bin"] == t) & (d["alta_exp"] == 1)).astype(int)
            if did_vars:
                m_es = pf.feols(f"share_jovem ~ {' + '.join(did_vars)} | cbo_4d + uf_periodo", data=d, vcov=VCOV_SPEC)
                print(f"Event study share_jovem — {label} conect.: {len(ts)} coeficientes")

### Verificação metodológica (conferência com o plano Etapa 3)

- **β₇ (triple_did):** Interpretar como a diferença do efeito da IA entre municípios de alta e baixa conectividade. β₇ &lt; 0 em salário real = efeito mais negativo onde a adoção de IA é viável.
- **Conectividade pré-tratamento:** Medida Jan–Out/2022 para evitar endogeneidade.
- **FE:** cbo_4d + uf_periodo absorvem choques estaduais; robustez com id_municipio + periodo.
- **Clustering:** Por id_municipio; robustez multiway (ocupação + município).
- **Comparação com Etapa 2:** Apresentar 3 como extensão — efeito médio concentrado nos municípios conectados.